In [36]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [37]:
## Load the ANN trained model,scaler pickle, onehot
model = load_model('model.h5')

## Load Encoders and scalers
with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo= pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [38]:
## Example input data

input_data = pd.DataFrame([{
    "CreditScore": 680,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 5,
    "Balance": 60000.0,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 75000.0
}])

In [39]:
geo_encoded = onehot_encoder_geo.transform(input_data[["Geography"]])

print(type(geo_encoded))
print(geo_encoded.shape)
print(onehot_encoder_geo.get_feature_names_out())

<class 'scipy.sparse._csr.csr_matrix'>
(1, 3)
['Geography_France' 'Geography_Germany' 'Geography_Spain']


In [40]:
# Apply OneHotEncoder to Geography column
geo_encoded = onehot_encoder_geo.transform(
    input_data['Geography'].values.reshape(-1, 1)
).toarray()
# Build DataFrame with correct shape
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

c:\Users\NIRAJ\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [41]:
print(input_data)

print(input_data["Gender"])
print(input_data["Gender"].iloc[0])
print(type(input_data["Gender"].iloc[0]))

   CreditScore Geography Gender  Age  Tenure  Balance  NumOfProducts  \
0          680    France   Male   40       5  60000.0              2   

   HasCrCard  IsActiveMember  EstimatedSalary  
0          1               1          75000.0  
0    Male
Name: Gender, dtype: object
Male
<class 'str'>


In [42]:
print(label_encoder_gender.classes_)
print(input_data["Gender"])
print(input_data["Gender"].dtype)
input_data["Gender"] = label_encoder_gender.transform(input_data["Gender"])

['Female' 'Male']
0    Male
Name: Gender, dtype: object
object


In [43]:
input_data = input_data.drop("Geography", axis=1)
input_data = pd.concat([input_data, geo_encoded_df], axis=1)

In [45]:
input_scaled = scaler.transform(input_data)

# Predict
prediction = model.predict(input_scaled)

if prediction[0][0] > 0.5:
    print("Customer is likely to churn.")
else:
    print("Customer is likely to stay.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Customer is likely to stay.
